# AFM Data – CrSBr Device 2

Reads Bruker NanoScope `.000` files, applies a **3-point plane calibration**, and
produces publication-ready topography maps with line-scan thickness measurements.

**Files:**
- `20260310_crsbr_device2_20um.000` — Device 2 scan

**Leveling.** `load_afm(file, level_points=[...])` does a Gwyddion-style three-point
level: a plane is fit exactly through three chosen substrate points and subtracted,
so those points sit at the same height (z = 0) and the substrate background is flat.
This avoids the bias of a whole-image plane fit (which the flake/electrodes tilt).
The three points are set in the "load" cell as `LEVEL_DEV2`.

The figure cell exposes `CROP_DEV2` (zoom window in µm) and `CUT_DEV2` (the two
line-cut endpoints in µm). Adjust them to move/resize the crop or the line scan;
the order of the two endpoints sets the profile's left-right direction. The line
scan is averaged over a band of `width_px` parallel pixel lines perpendicular to
the cut (set per figure, e.g. `width_px=10`) for a clean, low-noise profile.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar

# Bruker .000 reader lives in scripts/; numpy-only, replaces pySPM.
PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "scripts").is_dir())
SCRIPTS_DIR = PROJECT_ROOT / "scripts"
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))
from bruker_reader import read_bruker_height

DATA_DIR = PROJECT_ROOT / "data" / "AFM data"
OUT_DIR = PROJECT_ROOT / "output" / "AFM"
OUT_DIR.mkdir(parents=True, exist_ok=True)

FILE_DEV2 = DATA_DIR / "20260310_crsbr_device2_20um.000"


In [ ]:
from scipy.ndimage import uniform_filter1d, map_coordinates


def _sample_z(data: np.ndarray, sx: float, sy: float,
              x_um: float, y_um: float, radius: int = 4) -> float:
    """Mean height (nm) in a small window around an (x, y) µm point.

    Averaging a few pixels makes the 3-point reference robust to pixel noise.
    """
    ny, nx = data.shape
    j = int(round(x_um / sx * (nx - 1)))
    i = int(round(y_um / sy * (ny - 1)))
    win = data[max(0, i - radius):i + radius + 1,
               max(0, j - radius):j + radius + 1]
    return float(win.mean())


def level_three_point(data: np.ndarray, sx: float, sy: float,
                      points: list, radius: int = 4) -> np.ndarray:
    """Three-point plane calibration (Gwyddion-style 'three point level').

    A plane is fit EXACTLY through three chosen substrate points (each z taken
    as the mean of a small window) and subtracted, so those points end up at the
    same height, z = 0, and the surface they sit on becomes flat. Unlike a
    whole-image least-squares plane, this is not biased by the flake/electrodes,
    so the substrate background is properly flattened.

    points : list of three (x_um, y_um) tuples on the common reference surface.
    """
    if len(points) != 3:
        raise ValueError("three-point leveling needs exactly 3 points")
    ny, nx = data.shape
    rows, zs = [], []
    for (x_um, y_um) in points:
        rows.append([1.0, x_um / sx * (nx - 1), y_um / sy * (ny - 1)])
        zs.append(_sample_z(data, sx, sy, x_um, y_um, radius))
    a, b, c = np.linalg.solve(np.array(rows), np.array(zs))   # z = a + b*col + c*row
    jj, ii = np.meshgrid(np.arange(nx), np.arange(ny))
    return data - (a + b * jj + c * ii)


def load_afm(filepath: Path, level_points: list | None = None) -> dict:
    """Read and plane-level a Bruker NanoScope height image.

    Step (1) loads the first Height channel via bruker_reader (numpy-only).
    Step (2) removes tilt and sets the zero level:
      - If ``level_points`` (three (x, y) µm points on the bare substrate) is
        given, a 3-point plane calibration is applied: the plane through those
        points is subtracted, so the substrate they sit on becomes flat at z = 0.
      - Otherwise a whole-image least-squares plane is removed and the zero is
        set from the histogram peak of the lower half of the height distribution.

    Returns a dict with keys:
        data       – 2-D float array, height in nm
        sx, sy     – scan size in µm
        nx, ny     – pixel counts
    """
    data, sx, sy = read_bruker_height(filepath)   # height in nm; sx, sy in µm
    ny, nx = data.shape

    if level_points is not None:
        # --- 3-point plane calibration on chosen substrate points ---
        data = level_three_point(data, sx, sy, level_points)
    else:
        # --- whole-image plane fit + histogram-peak substrate zero (fallback) ---
        xv = np.linspace(0, 1, nx)
        yv = np.linspace(0, 1, ny)
        X, Y = np.meshgrid(xv, yv)
        A = np.c_[np.ones(nx * ny), X.ravel(), Y.ravel()]
        coef, *_ = np.linalg.lstsq(A, data.ravel(), rcond=None)
        data -= coef[0] + coef[1] * X + coef[2] * Y

        flat  = data.ravel()
        lower = flat[flat < np.percentile(flat, 50)]
        hist, bins = np.histogram(lower, bins=200)
        smooth = uniform_filter1d(hist.astype(float), size=7)
        pk = np.argmax(smooth)
        data -= 0.5 * (bins[pk] + bins[pk + 1])

    return dict(data=data, sx=sx, sy=sy, nx=nx, ny=ny)


def line_profile(afm: dict, p0: tuple, p1: tuple, n: int = 400,
                 width_px: int = 1):
    """Sample a straight line cut between two (x, y) points given in µm.

    Bilinear-interpolates the leveled height map along the segment p0 -> p1.
    With ``width_px`` > 1 the cut is widened into a band: ``width_px`` parallel
    lines, offset symmetrically by 1-pixel steps PERPENDICULAR to the cut, are
    sampled and averaged. This averages out per-pixel noise and gives a much
    cleaner profile than a single one-pixel line.

    Returns (distance_um, height_nm). Distance is measured from p0, so swapping
    p0 and p1 flips the profile's left-right orientation. (The perpendicular
    offset is taken in pixel space; valid because these scans have equal
    px/µm on both axes.)
    """
    data = afm['data']
    nx, ny = afm['nx'], afm['ny']
    sx, sy = afm['sx'], afm['sy']
    (x0, y0), (x1, y1) = p0, p1

    # endpoints in fractional pixel coords (col = x, row = y)
    c0, c1 = x0 / sx * (nx - 1), x1 / sx * (nx - 1)
    r0, r1 = y0 / sy * (ny - 1), y1 / sy * (ny - 1)
    cs = np.linspace(c0, c1, n)
    rs = np.linspace(r0, r1, n)

    # unit vector perpendicular to the cut, in pixel space
    dc, dr = c1 - c0, r1 - r0
    length = np.hypot(dc, dr)
    pc, pr = (-dr / length, dc / length) if length else (0.0, 0.0)

    # average over a band of `width_px` parallel offsets, centered on the cut
    offsets = np.linspace(-(width_px - 1) / 2, (width_px - 1) / 2, width_px)
    band = np.empty((width_px, n))
    for k, off in enumerate(offsets):
        band[k] = map_coordinates(
            data, np.vstack([rs + off * pr, cs + off * pc]),
            order=1, mode='nearest',
        )
    profile = band.mean(axis=0)

    dist = np.linspace(0, np.hypot(x1 - x0, y1 - y0), n)
    return dist, profile


def afm_figure(
    afm: dict,
    p0: tuple,
    p1: tuple,
    crop: tuple | None = None,
    z_lim: tuple | None = None,
    width_px: int = 1,
    scalebar_um: float = 2.0,
    cmap: str = 'afmhot',
    filename: Path | None = None,
) -> None:
    """Plot cropped AFM topography (left) and a straight line-cut profile (right).

    Parameters
    ----------
    afm          : dict from load_afm()
    p0, p1       : (x_um, y_um) endpoints of the line cut. Order sets the
                   profile direction (p0 at distance 0). Put the low side at p0
                   for an ascending profile, the high side at p0 for descending.
    crop         : (x0, x1, y0, y1) in µm to zoom the topography; None -> full scan
    z_lim        : (vmin, vmax) in nm; None -> 1st–99th percentile of the full scan
    width_px     : number of parallel pixel lines averaged perpendicular to the
                   cut (1 = single one-pixel line; 5–10 gives a clean band average)
    scalebar_um  : length of the white scale bar in µm
    cmap         : matplotlib colourmap
    filename     : save path (PNG); None -> no file saved
    """
    data = afm['data']
    sx, sy = afm['sx'], afm['sy']

    # --- Line cut (averaged over a band of width_px pixels) ---
    dist, profile = line_profile(afm, p0, p1, width_px=width_px)

    # --- Colour limits ---
    if z_lim is None:
        vmin = np.percentile(data, 1)
        vmax = np.percentile(data, 99)
    else:
        vmin, vmax = z_lim

    # --- Step height from the line profile ---
    h_sub   = np.median(profile[profile < np.percentile(profile, 20)])
    h_flake = np.median(profile[profile > np.percentile(profile, 80)])
    step_nm = h_flake - h_sub

    # ------------------------------------------------------------------ figure
    fig = plt.figure(figsize=(9, 4), dpi=300)
    gs  = gridspec.GridSpec(
        1, 2,
        width_ratios=[1.0, 0.85],
        wspace=0.42,
        left=0.04, right=0.97,
        bottom=0.12, top=0.94,
    )
    ax_img = fig.add_subplot(gs[0])
    ax_ls  = fig.add_subplot(gs[1])

    # --- Topography ---
    extent = [0, sx, sy, 0]   # [left, right, bottom, top]; y increases downward
    im = ax_img.imshow(
        data, cmap=cmap, vmin=vmin, vmax=vmax,
        extent=extent, origin='upper', aspect='equal',
    )
    if crop is not None:
        x0c, x1c, y0c, y1c = crop
        ax_img.set_xlim(x0c, x1c)
        ax_img.set_ylim(y1c, y0c)   # y increases downward, so top < bottom
    ax_img.plot(
        [p0[0], p1[0]], [p0[1], p1[1]],
        color='white', linewidth=0.9, linestyle='--', alpha=0.9,
    )
    ax_img.set_xticks([])
    ax_img.set_yticks([])

    # Colorbar
    cbar = fig.colorbar(im, ax=ax_img, fraction=0.046, pad=0.03)
    cbar.set_label(r'Height (nm)')

    # Scale bar (white, bottom-right of the cropped view)
    scalebar = AnchoredSizeBar(
        ax_img.transData,
        scalebar_um,
        f'{scalebar_um:g} µm',
        loc='lower right',
        pad=0.35,
        color='white',
        frameon=False,
        size_vertical=sy * 0.012,
    )
    ax_img.add_artist(scalebar)

    # --- Line scan ---
    ax_ls.plot(dist, profile, 'k-', linewidth=0.9)
    ax_ls.set_xlabel(r'Position ($\mu$m)')
    ax_ls.set_ylabel(r'Height (nm)')
    ax_ls.set_xlim(0, dist[-1])

    # Annotate step height
    if not np.isnan(step_nm):
        ax_ls.annotate(
            r'$\Delta h = $' + f'{step_nm:.1f} nm',
            xy=(0.04, 0.53), xycoords='axes fraction',
            va='top', ha='left',
        )

    if filename is not None:
        fig.savefig(filename, dpi=300)
    plt.show()

    if not np.isnan(step_nm):
        print(f'Step height from line scan: {step_nm:.2f} nm')

## Device 2 (2026-03-10)

In [ ]:
# 3-point plane calibration: three points on the bare substrate (right of the
# electrode), spread out and non-collinear, forced to the same z = 0.
LEVEL_DEV2 = [(13.5, 7.5), (17.0, 10.0), (14.5, 12.8)]   # (x, y) in µm

afm_dev2 = load_afm(FILE_DEV2, level_points=LEVEL_DEV2)
print(f"Scan size : {afm_dev2['sx']:.0f} × {afm_dev2['sy']:.0f} µm")
print(f"Z range   : {afm_dev2['data'].min():.1f}  to  {afm_dev2['data'].max():.1f} nm")
print(f"Z (p1–p99): {np.percentile(afm_dev2['data'],1):.1f}  to  {np.percentile(afm_dev2['data'],99):.1f} nm")

In [ ]:
# Crop to the electrode region; short horizontal cut from flake -> substrate.
CROP_DEV2 = (3, 19, 2, 18)        # (x0, x1, y0, y1) in µm
CUT_DEV2  = ((10.7, 9.0), (13.5, 8.0))   # (p0 high side, p1 low side) -> descending

afm_figure(
    afm_dev2,
    p0=CUT_DEV2[0],
    p1=CUT_DEV2[1],
    crop=CROP_DEV2,
    z_lim=(-3, 50),
    width_px=10,            # average the cut over a 10-pixel band
    scalebar_um=5,
    cmap='afmhot',
    filename=OUT_DIR / 'afm_device2_topography.png',
)